# [Re] Buda et al. 2019 — TCGA-LGG segmentation (single-notebook reproduction)

Runs the **entire** ReScience partial replication end to end and writes every metric and
figure needed to write the paper:

1. leakage test (must pass first) 2. build manifest 3. patient-level splits 4. train the
plain 4-level U-Net from scratch (5 folds, checkpoint/resume) 5. **per-patient** Dice/IoU/HD95
(headline vs paper 0.82/0.85) 6. robustness / quantization / distillation / cross-institution
7. `reports/metrics.json` + `reports/comparison.md` + all figures.

**Setup on Kaggle:** add the dataset `mateuszbuda/lgg-mri-segmentation` (Add Input), enable
GPU (P100 or T4). Everything writes to `/kaggle/working` so it survives *Save & Run All* and
resumes across the 12-hour session cap — just run the notebook again to continue training.

**Knobs** are in the next cell. Defaults finish the full pipeline comfortably within Kaggle
quota. Set `SMOKE=True` first to verify wiring in ~5 min.

In [ ]:
# ============================ KNOBS ============================
SMOKE          = False      # True = 2-patient wiring check (~5 min), ignores everything below
PRESET         = 'kfold5'   # 'kfold5' (default) or 'buda22' (paper-exact 22x5, ~3x compute)
EPOCHS         = 60         # per fold; lower to fit a single session, resume continues
IN_CHANNELS    = 3          # 3 = pre/FLAIR/post (headline); 1 = FLAIR-only (documented R9)
BATCHNORM      = True       # True = public-repo variant; False = strictly-paper net
FOLDS_TO_TRAIN = None       # None = all folds; or e.g. [0,1] to split across sessions

RUN_ROBUSTNESS       = True
RUN_QUANTIZE         = True
RUN_DISTILL          = True
RUN_CROSS_INSTITUTION= True  # trains 4 leave-one-site-out models (~ one extra fold each)

# GitHub repo with the src/lgg package (used if the code isn't already alongside the notebook).
REPO_URL = 'https://github.com/adithyabalakumar007-ai/mlrc-tracks-TCGA-LGG-comm.git'
# ===============================================================

In [ ]:
import os, sys, glob, subprocess, shutil, json, textwrap

# --- locate the src/lgg package: prefer local, else clone the repo -------------
def find_repo():
    for c in ['.', '/kaggle/working/repo', os.path.dirname(os.getcwd())]:
        if os.path.isdir(os.path.join(c, 'src', 'lgg')):
            return os.path.abspath(c)
    return None

REPO = find_repo()
if REPO is None:
    dst = '/kaggle/working/repo'
    if not os.path.isdir(dst):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, dst], check=True)
    REPO = dst
print('repo:', REPO)

# --- deps: Kaggle ships torch/pandas/scipy/sklearn; add medpy for HD95 ---------
try:
    import medpy  # noqa
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'medpy'], check=False)

SRC = os.path.join(REPO, 'src')
sys.path.insert(0, SRC)

def run(*cli_args):
    """Invoke the lgg CLI with the working config, streaming output live."""
    env = dict(os.environ, PYTHONPATH=SRC)
    cmd = [sys.executable, '-m', 'lgg.cli', '--config', CONFIG] + list(cli_args)
    print('>>>', ' '.join(cli_args))
    p = subprocess.run(cmd, env=env, cwd=WORK)
    if p.returncode != 0:
        raise RuntimeError(f'command failed: {cli_args}')

In [ ]:
# --- working dir + auto-detected dataset path + generated config.yaml ----------
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.path.join(REPO, '_run')
os.makedirs(WORK, exist_ok=True)

def detect_datapath():
    # Find the folder that directly contains TCGA_* patient subfolders.
    roots = ['/kaggle/input', os.path.join(REPO, 'data')]
    for root in roots:
        for dirpath, dirnames, _ in os.walk(root):
            if any(d.startswith('TCGA_') and os.path.isdir(os.path.join(dirpath, d)) for d in dirnames):
                return dirpath
    raise FileNotFoundError('Could not find TCGA_* folders. Add the lgg-mri-segmentation dataset.')

DATAPATH = detect_datapath()
print('datapath:', DATAPATH)

CONFIG = os.path.join(WORK, 'config.yaml')
cfg = f'''seed: 42
paths:
  datapath: {DATAPATH}
  manifest: {WORK}/reports/manifest.csv
  splits: {WORK}/splits/splits.json
  ckpt_dir: {WORK}/checkpoints
  reports: {WORK}/reports
  figures: {WORK}/figures
data:
  in_channels: {IN_CHANNELS}
splits:
  preset: {PRESET}
  seed: 42
  stratify_by_site: false
train:
  epochs: {EPOCHS}
  batch_size: 16
  lr: 0.001
  channels: reference
  batchnorm: {str(BATCHNORM).lower()}
  augment: true
  amp: true
  num_workers: 2
  foreground_bias: 0.0
  dice_weight: 1.0
  bce_weight: 1.0
  grad_clip: 0.0
  pretrained_encoder: false
distill:
  student_channels: micro
  epochs: 30
  temperature: 2.0
  alpha: 0.5
'''
with open(CONFIG, 'w') as fh:
    fh.write(cfg)
print(cfg)

In [ ]:
# --- R1: leakage test MUST pass before any training ---------------------------
env = dict(os.environ, PYTHONPATH=SRC)
subprocess.run([sys.executable, '-m', 'pytest', os.path.join(REPO, 'tests', 'test_no_leakage.py'), '-q'],
               env=env, cwd=REPO, check=True)

In [ ]:
# --- SMOKE path: 2-patient end-to-end wiring check, then stop -----------------
if SMOKE:
    run('prepare-data')
    run('smoke')
    raise SystemExit('SMOKE OK — set SMOKE=False for the full run.')

In [ ]:
# --- steps 1-2: manifest + patient-level splits ------------------------------
run('prepare-data')
run('make-splits')

In [ ]:
# --- step 3: train from scratch, per fold, resumable -------------------------
# Re-running this cell after a session dies continues each fold from its last
# epoch checkpoint (--resume); finished folds return immediately.
from lgg.data.splits import load_splits
folds = load_splits(f'{WORK}/splits/splits.json')['folds']
targets = FOLDS_TO_TRAIN if FOLDS_TO_TRAIN is not None else [f['fold'] for f in folds]
for fi in targets:
    run('train', '--fold', str(fi), '--resume')

In [ ]:
# --- step 4: HEADLINE per-patient Dice/IoU/HD95 vs paper 0.82/0.85 -----------
run('evaluate')

In [ ]:
# --- step 5: extensions (all reuse the same leakage-safe splits) -------------
if RUN_ROBUSTNESS:        run('run-robustness')
if RUN_QUANTIZE:          run('quantize')
if RUN_DISTILL:           run('distill')
if RUN_CROSS_INSTITUTION: run('cross-institution')

In [ ]:
# --- step 6: assemble reports/metrics.json + comparison.md + figures ---------
run('report')

In [ ]:
# --- show the headline table and figures inline ------------------------------
from IPython.display import Markdown, display
import matplotlib.pyplot as plt, matplotlib.image as mpimg
display(Markdown(open(f'{WORK}/reports/comparison.md').read()))
figs = sorted(glob.glob(f'{WORK}/figures/*.png'))
for fp in figs:
    plt.figure(figsize=(8, 5)); plt.imshow(mpimg.imread(fp)); plt.axis('off')
    plt.title(os.path.basename(fp)); plt.show()

In [ ]:
# --- bundle everything a co-author needs to write the paper ------------------
bundle = f'{WORK}/paper_artifacts'
os.makedirs(bundle, exist_ok=True)
for sub in ['reports', 'figures', 'splits']:
    if os.path.isdir(f'{WORK}/{sub}'):
        shutil.copytree(f'{WORK}/{sub}', f'{bundle}/{sub}', dirs_exist_ok=True)
shutil.make_archive(f'{WORK}/paper_artifacts', 'zip', bundle)
print('Wrote', f'{WORK}/paper_artifacts.zip')
print('Contents:')
for f in sorted(glob.glob(f'{bundle}/**/*', recursive=True)):
    if os.path.isfile(f):
        print(' ', os.path.relpath(f, bundle))